# 🚧 JalanKita — YOLOv8s Training (Scratch)
**Project**: Deteksi Kerusakan Jalan Sidoarjo  
**Classes**: lubang_besar, lubang_kecil, retak_kulit_buaya, retak_memanjang  
**Platform**: Amazon SageMaker Studio  
**Mode**: Training Baru dari Awal (Arsitektur Small) + Manual YAML

---
> ⚙️ **SEBELUM MULAI** — Isi bagian KONFIGURASI di Cell 2 terlebih dahulu!

## ⚙️ KONFIGURASI — WAJIB DIISI

In [ ]:
# ============================================================
#  WAJIB DIISI SEBELUM MENJALANKAN NOTEBOOK
# ============================================================

# --- Kaggle API Credentials ---
KAGGLE_USERNAME = "USERNAME_KAGGLE_ANDA"   # ← Ganti ini
KAGGLE_KEY      = "API_KEY_KAGGLE_ANDA"    # ← Ganti ini

# --- Dataset Private Kaggle ---
DATASET_SLUG = "USERNAME/NAMA-DATASET"     # ← Ganti ini

# --- Training Config ---
EPOCHS      = 200          # Target mAP maksimal
BATCH_SIZE  = 16           # Aman untuk model Small di GPU ml.g4dn.xlarge
IMG_SIZE    = 640          
PATIENCE    = 30           # Early stopping
SAVE_PERIOD = 5            # Simpan checkpoint tiap 5 epoch

# --- Output ---
PROJECT_NAME = "jalankita"
RUN_NAME     = "yolov8s_scratch_v1"

print("✅ Konfigurasi tersimpan!")
print(f"   Dataset  : {DATASET_SLUG}")
print(f"   Epochs   : {EPOCHS} | Batch: {BATCH_SIZE} | ImgSize: {IMG_SIZE}")

## 📦 STEP 1 — Install Dependencies

In [ ]:
import subprocess, sys

print("[1/3] Installing ultralytics...")
subprocess.run([sys.executable, "-m", "pip", "install", "ultralytics", "kaggle", "-q"], check=True)

print("[2/3] Cek GPU...")
result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], 
                        capture_output=True, text=True)
if result.returncode == 0:
    print(f"✅ GPU ditemukan: {result.stdout.strip()}")
else:
    print("⚠️  GPU tidak ditemukan! Pastikan instance GPU sudah dipilih.")

print("[3/3] Import libraries...")
import os, shutil, yaml
from pathlib import Path
from ultralytics import YOLO
import torch

print(f"✅ PyTorch: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")

## 🔑 STEP 2 — Setup Kaggle API

In [ ]:
import json

kaggle_dir = Path("/home/sagemaker-user/.kaggle")
kaggle_dir.mkdir(parents=True, exist_ok=True)

kaggle_json = {"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}
kaggle_path = kaggle_dir / "kaggle.json"

with open(kaggle_path, "w") as f:
    json.dump(kaggle_json, f)
os.chmod(kaggle_path, 0o600)

result = subprocess.run(["kaggle", "datasets", "list", "--mine"], capture_output=True, text=True)
if "403" in result.stderr or "401" in result.stderr:
    print("❌ Kaggle API Key salah! Cek USERNAME dan KEY di Konfigurasi.")
else:
    print("✅ Kaggle API terkoneksi!")

## 📥 STEP 3 — Download Dataset dari Kaggle

In [ ]:
DATASET_DIR = Path("/home/sagemaker-user/dataset")
DATASET_DIR.mkdir(parents=True, exist_ok=True)

print(f"📥 Mendownload dataset: {DATASET_SLUG}")
result = subprocess.run([
    "kaggle", "datasets", "download",
    "-d", DATASET_SLUG,
    "-p", str(DATASET_DIR),
    "--unzip"
], capture_output=True, text=True)

if result.returncode != 0:
    print(f"❌ Download gagal! {result.stderr}")
else:
    print("✅ Dataset berhasil didownload!")

## 🔧 STEP 4 — Validasi data.yaml (Prioritas Manual)

In [ ]:
# 1. Definisikan kemungkinan lokasi data.yaml
YAML_MANUAL_PATH = Path("/home/sagemaker-user/data.yaml")
yaml_files_in_dataset = list(DATASET_DIR.rglob("data.yaml")) if 'DATASET_DIR' in locals() else []

# 2. Tentukan prioritas
if YAML_MANUAL_PATH.exists():
    YAML_PATH = YAML_MANUAL_PATH
    print(f"💡 Menggunakan data.yaml MANUAL (Root Folder): {YAML_PATH}")
elif yaml_files_in_dataset:
    YAML_PATH = yaml_files_in_dataset[0]
    print(f"💡 data.yaml manual tidak ditemukan. Menggunakan dari dataset: {YAML_PATH}")
    
    # Update config path otomatis jika pakai bawaan dataset
    new_config = {
        "path":  str(YAML_PATH.parent.resolve()),
        "train": "train/images",
        "val":   "valid/images",
        "test":  "test/images",
        "nc":    4,
        "names": ["lubang_besar", "lubang_kecil", "retak_kulit_buaya", "retak_memanjang"]
    }
    with open(YAML_PATH, "w") as f:
        yaml.dump(new_config, f, default_flow_style=False, allow_unicode=True)
else:
    YAML_PATH = YAML_MANUAL_PATH # Fallback untuk validasi error di bawah

# 3. Tampilkan isi YAML yang terpilih untuk konfirmasi
if YAML_PATH.exists():
    print("\n📄 Isi file data.yaml yang akan dipakai:")
    with open(YAML_PATH, "r") as f:
        print(f.read())
else:
    print("❌ File data.yaml tidak ditemukan di mana pun!")

## 💡 STEP 5 — Informasi Model

In [ ]:
print("💡 Mode: Latih ulang dari Scratch (Awal)")
print("   Kita akan menggunakan model dasar 'yolov8s.pt' (Small).")
print("   Pengecekan last.pt dilewati agar arsitektur model sesuai kebutuhan backend.")

## 🚀 STEP 6 — Mulai Training YOLOv8s

In [ ]:
# Validasi akhir sebelum run
errors = []
if not YAML_PATH.exists():
    errors.append("data.yaml belum ada (Pastikan sudah di-drag & drop ke folder utama)")
if not torch.cuda.is_available():
    errors.append("GPU tidak terdeteksi")

if errors:
    print("❌ Training dibatalkan! Ada yang belum siap:")
    for e in errors:
        print(f"   - {e}")
else:
    print("✅ Semua siap! Memulai training YOLOv8 Small dari awal...")
    print(f"   Model   : yolov8s.pt (Pretrained Resmi)")
    print(f"   Dataset : {YAML_PATH}")
    print(f"   Epochs  : {EPOCHS} | Batch: {BATCH_SIZE}")
    print("-" * 50)

    model = YOLO("yolov8s.pt")

    results = model.train(
        data        = str(YAML_PATH),
        epochs      = EPOCHS,
        imgsz       = IMG_SIZE,
        batch       = BATCH_SIZE,
        resume      = False,       # Latih model baru dari awal
        device      = 0,
        patience    = PATIENCE,
        save_period = SAVE_PERIOD,
        project     = f"/home/sagemaker-user/runs/{PROJECT_NAME}",
        name        = RUN_NAME,
        exist_ok    = True,
        # Parameter Augmentasi untuk Jalan Rusak (Objek Kecil)
        hsv_h       = 0.015,
        hsv_s       = 0.7,
        hsv_v       = 0.4,
        flipud      = 0.5,
        fliplr      = 0.5,
        mosaic      = 1.0,
        scale       = 0.5,         # Optimasi retak kecil
        copy_paste  = 0.3          # Variasi lokasi retak
    )

    print("\n" + "="*50)
    print("✅ TRAINING SELESAI!")
    print(f"   mAP50     : {results.results_dict.get('metrics/mAP50(B)', 0):.4f}")
    print(f"   mAP50-95  : {results.results_dict.get('metrics/mAP50-95(B)', 0):.4f}")

## 💾 STEP 7 — Backup Hasil Training

In [ ]:
import time
from datetime import datetime

RUNS_DIR    = Path(f"/home/sagemaker-user/runs/{PROJECT_NAME}/{RUN_NAME}")
BACKUP_DIR  = Path("/home/sagemaker-user/backup")
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

timestamp   = datetime.now().strftime("%Y%m%d_%H%M")
backup_name = f"jalankita_{RUN_NAME}_{timestamp}"
backup_path = BACKUP_DIR / backup_name

print("💾 Menyimpan hasil training...")

if RUNS_DIR.exists():
    shutil.make_archive(str(backup_path), "zip", str(RUNS_DIR))
    zip_size = (Path(str(backup_path) + ".zip")).stat().st_size / 1e6
    print(f"✅ Backup tersimpan: {backup_path}.zip ({zip_size:.1f} MB)")

    weights_dir = RUNS_DIR / "weights"
    for pt_name in ["last.pt", "best.pt"]:
        src = weights_dir / pt_name
        if src.exists():
            shutil.copy(src, Path("/home/sagemaker-user") / pt_name)
            print(f"✅ {pt_name} disalin ke /home/sagemaker-user/{pt_name}")
else:
    print("❌ Folder runs tidak ditemukan.")

## 📊 STEP 8 — Evaluasi Model (Opsional)

In [ ]:
best_pt = Path("/home/sagemaker-user/best.pt")
if not best_pt.exists():
    best_pt = RUNS_DIR / "weights" / "best.pt"

if best_pt.exists():
    print("📊 Mengevaluasi model terbaik...")
    model_eval = YOLO(str(best_pt))
    metrics    = model_eval.val(data=str(YAML_PATH), device=0)

    print("\n📈 Hasil Evaluasi per Kelas:")
    class_names = ["lubang_besar", "lubang_kecil", "retak_kulit_buaya", "retak_memanjang"]
    for i, name in enumerate(class_names):
        try:
            ap = metrics.box.ap50[i]
            print(f"   {name:25s}: AP50 = {ap:.4f}")
        except:
            pass
else:
    print("⚠️  best.pt belum ada.")